# 02 · FunnyBirds + CBM — grounding & mechanism

**Claim under test (CBM).** Concepts are a faithful bottleneck: a part concept
reflects *its part*. **Backwash** is the failure where a part concept instead reads
*species / the rest of the bird* — it reports a part it cannot see.

Two probes, read together:
- **Deletion grounding** (causal): remove a present part, does its concept stay high?
  → `grounding/funnybirds-cbm-s*.parquet`.
- **Species probe** (mechanism): is the bottleneck a species code?
  → `species_probe/funnybirds-cbm-s*.json`.

*Reference (method source): `fb_cbm_renderer_swap_v2.ipynb` (z-ordering / occlusion),
`fb_cbm_counterfactual.ipynb` §6 (species probe).*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"
plt.rcParams["figure.dpi"]=120
def parse_stem(stem):
    m=re.match(r"^funnybirds-(vanilla|cbm|mcbm)(?:-g([0-9p]+))?-s(\d+)$", stem)
    if not m: return None
    gamma=float(m.group(2).replace("p",".")) if m.group(2) else np.nan
    return m.group(1), gamma, int(m.group(3))
def need(p, how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok


## What a CBM is, and what `z` and `c_preds` are
A Concept Bottleneck Model routes the classifier through named concepts:

`image x → encoder p(z|x) → z → concept head q(c|z) → c_preds → label head → y`

- **`z`** — the *bottleneck representation*: a per-concept latent the encoder reads
  from the image (26 slots here, one per concept). Everything about the image that
  reaches the label must pass through `z`.
- **`c_preds`** — the concept *predictions*: the concept head applied to `z`, one
  probability per concept (e.g. P(tail_5 present)). This is the human-readable layer
  the model exposes and that test-time intervention edits.
- **`y`** — the class, predicted from `z` / the concepts.

Backwash lives in the **encoder→`z`** step: does `z_j` (hence `c_preds_j`) read *its
part's pixels*, or the species / rest of the bird? The probes below test exactly that.

## 0 · Training sanity & overfitting — which epoch to trust
Per-epoch held-out predictions live in `results/funnybirds-cbm/<seed>/predictions/
epoch_*.pth`. Plot task + concept accuracy vs epoch: **peak then decline** → use the
peak (overfit past it); **plateau** → any late epoch is fine (extra epochs are just
wasted compute). Also settles epoch_100 vs epoch_150.

In [ ]:
import torch
SEED = 1
preds = REPO/"external"/"minimal_cbm"/"results"/"funnybirds-cbm"/str(SEED)/"predictions"
def _acc(pth):
    d = torch.load(pth, map_location="cpu", weights_only=False)
    yp, y = d["y_preds"], d["y"]
    ta = (yp.argmax(-1)==y).float().mean().item() if yp.ndim>1 else (yp==y).float().mean().item()
    ca = None
    if d.get("c_preds") is not None and d.get("c") is not None:
        cp = d["c_preds"]; cp = cp[...,0] if cp.ndim==3 else cp
        ca = ((cp>=0.5).float()==d["c"]).float().mean().item()
    return ta, ca
files = sorted(glob.glob(str(preds/"epoch_*.pth")), key=lambda p:int(re.findall(r"epoch_(\d+)",p)[0]))
if not files:
    print(f"[pending] no per-epoch predictions in {preds} (trainer save=False).")
else:
    R = pd.DataFrame([(int(re.findall(r"epoch_(\d+)",f)[0]), *_acc(f)) for f in files],
                     columns=["epoch","task","concept"]).sort_values("epoch")
    best = int(R.loc[R.task.idxmax(),"epoch"]); display(R.round(4))
    fig,ax=plt.subplots(figsize=(6,3.6))
    ax.plot(R.epoch, R.task, "o-", color=CBM_C, label="task acc (val)")
    if R.concept.notna().any(): ax.plot(R.epoch, R.concept, "s--", color="#5B8C5A", label="concept acc (val)")
    ax.axvline(best, ls=":", color="k"); ax.set_xlabel("epoch"); ax.set_ylabel("val accuracy")
    ax.set_title(f"Training curve — best task-acc epoch = {best}"); ax.legend()
    drop = R.task.max() - R.task.iloc[-1]
    print(f"best task epoch={best} (acc {R.task.max():.3f}); final-epoch drop from peak = {drop:+.3f}")
    print("VERDICT:", "plateau -> no overfit; late epochs are wasted compute" if abs(drop)<0.01
          else f"peaks at {best} then declines -> use epoch_{best}")

## 1 · Deletion grounding — per-part `retained_frac`
For each present part: delete it, read `P(species-typical concept)`.
`retained_frac = P(removed)/P(intact)` ∈ [0,1]. tail stays high; wing/foot/beak/eye
collapse to ~0. (See the bottom cell for why this is a backwash measure.)

In [ ]:
SEED = 1
gp = CURATED/"grounding"/f"funnybirds-cbm-s{SEED}.parquet"
if need(gp, "bash analysis/grounding_sweep.sh"):
    g = pd.read_parquet(gp)
    if 'changed_frac' in g.columns: g = g[g['changed_frac']>1e-3]  # visible-only
    def agg(d):
        pi, pr = d.p_intact.mean(), d.p_removed.mean()
        return pd.Series({"n":len(d),"p_intact":pi,"p_removed":pr,
                          "retained_frac": pr/pi if pi>1e-6 else np.nan})
    per = g.groupby("part")[["p_intact","p_removed"]].apply(agg).sort_values("retained_frac", ascending=False)
    display(per.round(3))
    fig,ax=plt.subplots(figsize=(6,3.2))
    ax.bar(per.index, per.retained_frac, color=CBM_C)
    ax.set_ylabel("retained_frac\n(P concept removed / intact)"); ax.set_ylim(0,1)
    ax.set_title("FunnyBirds · CBM · removed-part concept retention"); plt.xticks(rotation=30,ha="right")
    OVERALL = g.p_removed.mean()/g.p_intact.mean()
    print(f"OVERALL retained_frac = {OVERALL:.3f}")

## 2 · Species-identity probe — is the bottleneck a class code?
Linear probe (5-fold CV) recovering **species** from the bottleneck. `species←c_preds`
= how much the reported 26-d concept vector alone pins the species → a part concept is
answerable by *class-lookup* (the standing opportunity for backwash). Chance = 1/50 = 0.02.
*(Caveat: with clean concepts, `species←c_preds≈1` is partly tautological; the
informative numbers are the per-part ones.)*

In [ ]:
sp = CURATED/"species_probe"/f"funnybirds-cbm-s{SEED}.json"; PART=None
if need(sp, "bash analysis/grounding_sweep.sh"):
    S = json.loads(sp.read_text()); ch = S["chance"]
    print(f"chance={ch:.3f} | species<-z {S['species_from_z']['acc']:.3f} | "
          f"species<-c_preds {S['species_from_cpreds']['acc']:.3f}")
    fig,ax=plt.subplots(1,2,figsize=(10,3.2))
    ax[0].bar(["z","c_preds"], [S['species_from_z']['acc'], S['species_from_cpreds']['acc']], color=CBM_C)
    ax[0].axhline(ch, ls="--", color="k", label="chance"); ax[0].set_ylim(0,1)
    ax[0].set_title("species recoverable from bottleneck"); ax[0].legend()
    PART = pd.DataFrame(S["species_from_part_cpreds"]).T; PART["acc"]=PART["acc"].astype(float)
    PART = PART.sort_values("acc", ascending=False)
    ax[1].bar(PART.index, PART.acc, color=CBM_C); ax[1].axhline(ch,ls="--",color="k")
    ax[1].set_title("species from EACH part's concepts alone")
    plt.setp(ax[1].get_xticklabels(),rotation=30,ha="right"); display(PART)

## 3 · Line them up — does per-part retention track the mechanism?
Per part: measured **`retained_frac`** vs **species-code** (species-from-that-part) vs
**n_variants**. Correlation, not proof — the deletion test is the causal probe; this
shows whether the mechanism co-locates. (Note: wing has high species-code but low
retention — encoding species is *necessary* for backwash, not *sufficient*; the part
must also be the shortcut the label head uses.)

In [ ]:
if 'per' in dir() and PART is not None:
    M = per[["retained_frac"]].join(PART[["acc","n_variants"]].rename(columns={"acc":"species_code"}))
    M["n_variants"]=M["n_variants"].astype(float); display(M.round(3))
    fig,ax=plt.subplots(1,2,figsize=(10,3.4))
    for k,(x,xl) in enumerate({"species_code":"species from part's concepts","n_variants":"# concept variants"}.items()):
        ax[k].scatter(M[x], M.retained_frac, color=CBM_C)
        for p,r in M.iterrows(): ax[k].annotate(p,(r[x],r.retained_frac),fontsize=8)
        ax[k].set_xlabel(xl); ax[k].set_ylabel("retained_frac (removed part)")
    plt.tight_layout()
    print("A part high on BOTH species-code and retained_frac = the mechanism realized (tail).")
else:
    print("[pending] needs sections 1 and 2 to have produced outputs.")

## Takeaway
CBM concepts are near-perfect on-distribution, yet a **removed** part's concept is
retained (`retained_frac` high for tail), concentrated where the bottleneck most
encodes species. Notebook 03 asks whether MCBM's minimality removes this.

## How `retained_frac` is read as a backwash measurement
No table or axis here is labelled "backwash" — the number we actually compute is the
literal **`retained_frac = P(typical concept | part removed) / P(typical concept | part
intact)`**. Reading it as backwash:

- A **grounded** concept must *see its part*. Delete the part → nothing to see → its
  probability should collapse → `retained_frac → 0`.
- A **backwashed** concept infers its part from the species / rest of the bird. Deleting
  the part changes nothing it relied on → the probability stays up → `retained_frac → 1`.

So `retained_frac` is the operational metric; "concept–class backwash" is the
*interpretation* of a high `retained_frac`. We keep the two separate so the definition
is explicit and not conflated with other uses of the word 'backwash'.